# Milestone 5 lab, notebook 7 of 7: The bad morning

## Where we are

| Notebook | What you do |
|---|---|
| 1. A clock is not enough | Start two steps by the clock, and get yesterday's report |
| 2. Tasks and arrows | Read the DAG, and run it once |
| 3. Which day is this run for? | The logical date, the data interval, and a run you trigger by hand |
| 4. When a task fails | Break the API, then break the data, and watch what Airflow does |
| 5. The days you missed | Run a week with one command. Run one day again |
| 6. Every five minutes | A lab: the same pipeline as a micro-batch |
| **7. The bad morning** (you are here) | The drill: the API is down, then a page comes twice |

This is the last notebook of the project. It is a drill.

A pipeline is not finished when it works. It is finished when it survives a bad morning with nobody helping. So here is one: **the order system is down at 07:00. When it comes back, it sends one page twice.**

In [1]:
import time

from lab_helpers import MILESTONE_DIR, api, at, report_is_right, schedule_warehouse, show_doc, show_runs, start_fresh, table

from daily_orders import pipeline

at.wait_until_ready()
start_fresh()

Airflow has no runs. The warehouse is empty. The order system is back at its first morning.


## Step 1: A normal week

First, five good mornings, with one command.

In [2]:
at.backfill("2026-06-07", "2026-06-11")
show_runs()

,business_day,started_by,state
0,2026-06-07,backfill,success
1,2026-06-08,backfill,success
2,2026-06-09,backfill,success
3,2026-06-10,backfill,success
4,2026-06-11,backfill,success


In [3]:
print("The report equals the answer key for the morning of 12 June:", report_is_right("2026-06-12"))

The report equals the answer key for the morning of 12 June: True


## Step 2: The bad morning

It is Saturday 13 June, just before 07:00. Two things are about to go wrong.

1. The order system will answer its next **seven** calls with an error. That is more than one try of the load survives.
2. After that, it will serve **one page twice**.

In [4]:
api.post("/admin/faults", json={"kind": "error_500", "count": 7})
api.post("/admin/faults", json={"kind": "repeat_page", "count": 1})
print("Planned faults:", api.post("/admin/faults", json={"kind": "rate_limit", "count": 0}).json())

Planned faults: {'error_500': 7, 'rate_limit': 0, 'repeat_page': 1}


07:00. The run for Friday 12 June starts. Nobody is at their desk. Watch.

In [5]:
at.trigger("2026-06-12", wait=False)

seen, started = [], time.time()
while True:
    states = at.task_states("2026-06-12")
    line = "  ".join(f"{task.split('_')[0]}: {states.get(task) or 'waiting'}" for task in
                     ["set_source_clock", "extract_and_load", "transform_and_test", "publish_report"])
    if not seen or seen[-1] != line:
        seen.append(line)
        print(f"+{time.time() - started:3.0f}s   {line}")
    if all(run["state"] in ("success", "failed") for run in at.runs()):
        break
    time.sleep(2)

+  2s   set: success  extract: running  transform: waiting  publish: waiting


+ 19s   set: success  extract: up_for_retry  transform: waiting  publish: waiting


+ 36s   set: success  extract: running  transform: waiting  publish: waiting


+ 42s   set: success  extract: success  transform: success  publish: success


Read the middle column, `extract`.

- `running`: the load starts. Inside it, the code of milestone 2 tries five times, waiting longer each time. The order system fails all five.
- `up_for_retry`: Airflow sees the task fail. It has tries left, so it waits 20 seconds.
- `running`: Airflow starts the task again. Two more failures, and then the order system answers. One page arrives twice.
- `success`. The arrows let `transform` start, then `publish`.

## Step 3: Is the report right?

In [6]:
print("The report equals the answer key for the morning of 13 June:", report_is_right("2026-06-13"))

The report equals the answer key for the morning of 13 June: True


In [7]:
con = schedule_warehouse()
table(con, """
    SELECT l.run_id, l.rows_landed, l.api_calls, l.retries AS retries_inside_the_code, l.tests_passed, l.tests_failed,
           (SELECT count(DISTINCT payload->>'order_id') FROM raw.orders_landing r WHERE r.run_id = l.run_id) AS different_orders
    FROM ops.load_runs l ORDER BY l.loaded_at DESC LIMIT 1
""")

,run_id,rows_landed,api_calls,retries_inside_the_code,tests_passed,tests_failed,different_orders
0,2026-06-13_1,118,7,2,31,0,93


Everything this project built is in that one row.

| What you see | What made it harmless | Built in |
|---|---|---|
| The API failed, and the load waited and tried again | Retry with backoff, inside the code | Milestone 2 |
| The whole task failed, and was started again | A task retry in Airflow | Milestone 5 |
| The first try landed nothing, and the watermark did not move | One transaction for rows and watermark | Milestone 2 |
| More rows landed than there are orders: a page came twice | Staging keeps one row per order. The write is idempotent | Milestones 3 and 4 |
| The report is right | 31 tests on the tables passed before it was published | Milestone 4 |
| Nobody did anything | A schedule, and dependencies between tasks | Milestone 5 |

In [8]:
print(pipeline.report_text(con))
con.close()

Arden daily orders. Report for the morning of Sat 2026-06-13

  Yesterday   Fri 2026-06-12    26 orders      $704.74
  Day before  Thu 2026-06-11    30 orders      $762.24
  Change                         -4 orders      -$57.50

  Earlier days: no change in this run.


The manager opens the report at 08:00. It is right. She never knows there was a bad morning.

In [9]:
show_doc("LLD.md", "7. The bad morning drill")

**From `docs/LLD.md`**

> ## 7. The bad morning drill
> 
> *(There is a diagram here. Open `docs/LLD.md` to see it.)*
> 
> Measured in the test `test_the_bad_morning_the_api_is_down_then_a_page_comes_twice`: the task goes `running`, `up_for_retry`, `running`, `success`. The run lands more rows than it has orders, and the report equals the answer key.
> 
> Two layers of retry are at work, and they are different:
> 
> | Layer | Where | It handles |
> |---|---|---|
> | In the code (milestone 2) | Inside one call to the API: 5 tries in about 15 seconds | A hiccup of a few seconds |
> | In Airflow (this milestone) | The whole task, again, after 20 seconds, up to 3 more times | An outage of a minute or two |

## Nothing is still wrong

Every milestone of this project ended with a line that began "Still wrong". This one does not.

In [10]:
show_doc("LLD.md", "9. Nothing is still wrong")

**From `docs/LLD.md`**

> ## 9. Nothing is still wrong
> 
> Every "still wrong" line of this project has been closed.
> 
> | It was still wrong that | Closed in |
> |---|---|
> | It reloads every order every day, and hits the rate limit | Milestone 2: an incremental load |
> | A changed order is in the table twice, and a second run makes it worse | Milestone 3: `MERGE`, proved by tests |
> | All the logic is in one script, and nothing checks the tables | Milestone 4: models and table tests |
> | A person must remember to run it | Milestone 5: a schedule |
> 
> A real company would add more. These belong to other projects of the series:
> 
> | Next | What it adds |
> |---|---|
> | **Alerts** | A message to a person when a run fails, and when a run did not happen at all |
> | **A quality gate** | Checks with thresholds, findings a person reviews, and a decision to publish or hold |
> | **A dashboard** | The report as charts, on top of `marts.daily_orders` |

A real company would build more around this pipeline: alerts when a run fails or does not happen, a quality gate where a person approves findings, and a dashboard on top of the report. Those belong to other projects of the D8Loop series. They are not missing pieces of this one.

## The words of milestone 5

| Word | In one or two sentences | You saw it in |
|---|---|---|
| **Schedule**, **cron** | A schedule starts a run at a time. Cron's five fields are the usual way to write one | Notebook 1 |
| **Orchestration** | Running the steps in the right order, at the right time, and watching them | Notebook 1 |
| **Task**, **dependency**, **DAG** | One step. A rule that one task needs another to finish first. The tasks and the arrows together | Notebook 2 |
| **Data interval**, **logical date** | The slice of time a run covers. The day a run is for | Notebook 3 |
| **Trigger** | Starting a run by hand, for a day you name | Notebook 3 |
| **Retry** (of a task) | Running a failed task again after a wait | Notebook 4 |
| **Catchup** | A scheduler running missed days by itself | Notebook 5 |
| **Backfill** | Running the job for a range of past days | Notebook 5 |
| **Replay** | Running a finished day again | Notebook 5 |
| **Micro-batch** | The same batch, with a very small slice | Notebook 6 |

The project's [`GLOSSARY.md`](../../../GLOSSARY.md) keeps every word of all five milestones, with an interview line for each.

## What you can now do

- Turn a business question into a table with a clear grain.
- Prove a pipeline is idempotent with a test, not with a claim.
- Re-run a past day without touching the others.
- Read a failed run, and say which stage failed and whether a re-run is safe.
- Explain each decision in an interview, with a real example from Arden.

**Say it in an interview:** "I built a daily batch pipeline from an order API to a warehouse. It loads incrementally with a watermark and a lookback, lands raw, builds dbt models with table tests, and runs on Airflow. I can show it is safe: I have a drill where the API is down at seven and then sends a page twice. The task retries, the duplicate page changes nothing, and the report still equals the answer key."

**You have finished the project.** Open [`milestone_5_recap.html`](../milestone_5_recap.html) in a browser. Then try the live page: from the project folder, `uv run python pipeline_live/server.py`.